In [ ]:
!pip install requests pandas nltk unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.5 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from unidecode import unidecode
import numpy as np

In [ ]:
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# **RF–01 Recoleccion de datos**


In [9]:
token_github = "TOKEN_AQUI"

repositorio = "scikit-learn/scikit-learn"

cabeceras = {}
if token_github:
    cabeceras["Authorization"] = f"token {token_github}"

def obtener_issues(repositorio):
    url = f"https://api.github.com/repos/{repositorio}/issues"
    respuesta = requests.get(url,headers=cabeceras, params={"state": "all", "per_page": 100})
    respuesta.raise_for_status()

    return [
        {
            "id": i["id"],
            "numero": i["number"],
            "tipo": "issue",
            "titulo": i["title"],
            "texto": i["body"] or ""
        }
        for i in respuesta.json()
        if "pull_request" not in i
    ]


def obtener_pull_requests(repositorio):
    url = f"https://api.github.com/repos/{repositorio}/pulls"
    respuesta = requests.get(
        url,
        headers=cabeceras,
        params={"state": "all", "per_page": 100}
    )
    respuesta.raise_for_status()

    return [
        {
            "id": pr["id"],
            "numero": pr["number"],
            "tipo": "pull_request",
            "titulo": pr["title"],
            "texto": pr["body"] or ""
        }
        for pr in respuesta.json()
    ]


def obtener_comentarios(repositorio, numero):
    url = f"https://api.github.com/repos/{repositorio}/issues/{numero}/comments"
    respuesta = requests.get(url, headers=cabeceras)
    respuesta.raise_for_status()

    return [
        {
            "id": c["id"],
            "numero": numero,
            "tipo": "comentario",
            "titulo": "",
            "texto": c["body"]
        }
        for c in respuesta.json()
    ]


datos = []

issues = obtener_issues(repositorio)
prs = obtener_pull_requests(repositorio)

for item in issues + prs:
    datos.append(item)
    comentarios = obtener_comentarios(repositorio, item["numero"])
    datos.extend(comentarios)

datos_df = pd.DataFrame(datos)
datos_df.to_csv("datos_github.csv", index=False, encoding="utf-8")
datos_df


,id,numero,tipo,titulo,texto
0,3910457181,33239,issue,[CODE CONTRIBUTION] Add incremental learning u...,### Describe the workflow you want to enable\n...
1,3910456758,33238,issue,[CODE CONTRIBUTION] Add class imbalance metric...,### Describe the workflow you want to enable\n...
2,3910456259,33237,issue,[CODE CONTRIBUTION] Add model performance degr...,### Describe the workflow you want to enable\n...
3,3910455738,33236,issue,[CODE CONTRIBUTION] Add feature correlation de...,### Describe the workflow you want to enable\n...
4,3910454541,33235,issue,[CODE CONTRIBUTION] Add robustness metrics for...,### Describe the workflow you want to enable\n...
...,...,...,...,...,...
345,3767758613,33101,comentario,,"Hi @lucyleeow,\r\n\r\nI have updated the PR de..."
346,3767780542,33101,comentario,,> Please do not ping (@) maintainers.
347,3769606638,33101,comentario,,Understood. Apologies for the ping.
348,3832675062,33101,comentario,,Thank you for your interest in contributing to...


# **RF–02 Preprocesamiento de texto**

In [ ]:
stopwords = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")

In [ ]:
def limpiar_ruido(texto):

    texto = re.sub(r"\b[b-df-hj-np-tv-z]{8,}\b", "", texto)


    texto = re.sub(r"\b[a-f0-9]{10,}\b", "", texto)

    palabras = []

    for p in texto.split():
        if len(p) > 15:
            continue

        if len(p) >= 8:
            diversidad = len(set(p)) / len(p)
            if diversidad < 0.4:
                continue

        palabras.append(p)

    return " ".join(palabras)


def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""

    texto = texto.lower()
    texto = unidecode(texto)
    texto = re.sub(r"http\S+", "", texto)
    texto = re.sub(r"[^a-z\s]", "", texto)
    texto = limpiar_ruido(texto)

    palabras = [
        stemmer.stem(p)
        for p in texto.split()
        if p not in stopwords and len(p) > 2
    ]

    return " ".join(palabras)


datos_df["texto_limpio"] = datos_df["texto"].apply(limpiar_texto)
datos_df[["tipo", "titulo", "texto_limpio"]]



,tipo,titulo,texto_limpio
0,issue,BUG undefined `n_classes` for custom estimator...,describ bug give evid userfac impact introduc ...
1,comentario,,sure best option look isregressor iscluster et...
2,comentario,,sinc break would favour quick solut first ill ...
3,comentario,,first last item suggest lucyleeow would rather...
4,comentario,,isregressor enough make estim type explicit qu...
...,...,...,...
327,comentario,,sorri confus look output seem odd remov condit...
328,comentario,,unless alway run parallel whether azur paralle...
329,comentario,,diff may bit confus current behaviour correct ...
330,comentario,,okay thank point old enough run see right tri ...


# **RF–03 Representacion del texto**


In [ ]:
vectorizador= TfidfVectorizer(
    ngram_range=(1, 2),
    max_df=0.9,
    min_df=5
)

matriz= vectorizador.fit_transform(datos_df["texto_limpio"])

print("Dimension de la matriz TF-IDF:", matriz.shape)

Dimension de la matriz TF-IDF: (332, 759)


# **RF–04 Analisis de sentimiento**

In [ ]:
analizador = SentimentIntensityAnalyzer()

def etiquetar_sentimiento(texto):
    puntaje = analizador.polarity_scores(texto)["compound"]
    if puntaje >= 0.05:
        return "positivo"
    elif puntaje <= -0.05:
        return "negativo"
    else:
        return "neutral"

datos_df["sentimiento"] = datos_df["texto"].fillna("").apply(etiquetar_sentimiento)

X_train, X_test, y_train, y_test = train_test_split(
    matriz,
    datos_df["sentimiento"],
    test_size=0.2,
    random_state=42,
    stratify=datos_df["sentimiento"]
)

modelo = LogisticRegression(max_iter=1000, class_weight='balanced')
modelo.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
predicciones = modelo.predict(X_test)

print("F1-macro:",
      f1_score(y_test, predicciones, average="macro"))

print("\nReporte de clasificacion:\n")
print(classification_report(y_test, predicciones))

F1-macro: 0.7375437232802177

Reporte de clasificacion:

              precision    recall  f1-score   support

    negativo       0.62      0.67      0.65        15
     neutral       0.70      0.70      0.70        10
    positivo       0.88      0.86      0.87        42

    accuracy                           0.79        67
   macro avg       0.73      0.74      0.74        67
weighted avg       0.79      0.79      0.79        67



# **RF–05 Identificacion de temas**

In [ ]:
k = 4

modelo_kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = modelo_kmeans.fit_predict(matriz)

datos_df["tema"] = clusters

print("Distribución de temas:")
print(datos_df["tema"].value_counts())


Distribución de temas:
tema
1    183
0     90
2     40
3     19
Name: count, dtype: int64


In [ ]:
terminos = vectorizador.get_feature_names_out()

for i in range(k):
    print(f"\nTema {i}:")
    indices = modelo_kmeans.cluster_centers_[i].argsort()[-15:]
    print(", ".join(terminos[j] for j in indices))


Tema 0:
valid, paramet, case, refer, chang, test, plot, class, valu, exampl, data, import, cluster, use, estim

Tema 1:
use, lestev, close see, let, fail, run, think, pleas, ogrisel, work, build, close, see, issu, thank

Tema 2:
review, maintain, exampl, creat, thank contribut, generat, thank, chang, pleas, fix, see, request, pull, pull request, contribut

Tema 3:
note task, file note, new branch, fail creat, file, detect issu, detail, import, detect, note, lint issu, issu, branch, lint, ruff


# **RF–06 Similitud textual**

In [ ]:
pd.set_option("display.max_colwidth", 200)

def mensajes_mas_similares(indice_mensaje, top_k=5):
    print("MENSAJE DE REFERENCIA")
    print("Tipo:", datos_df.loc[indice_mensaje, "tipo"])
    print("Titulo:", datos_df.loc[indice_mensaje, "titulo"])
    print("Texto:", datos_df.loc[indice_mensaje, "texto_limpio"])
    print("-" * 60)

    vector_ref = matriz[indice_mensaje]

    similitudes = cosine_similarity(vector_ref, matriz).flatten()
    similitudes[indice_mensaje] = -1

    indices_similares = np.argsort(similitudes)[-top_k:][::-1]

    resultado = datos_df.loc[indices_similares, [
        "tipo", "titulo", "texto_limpio", "tema"
    ]].copy()

    resultado["similitud"] = similitudes[indices_similares]
    resultado = resultado.sort_values(by="similitud", ascending=False)
    resultado = resultado[resultado["similitud"] >= 0.1]

    return resultado.reset_index(drop=True)


mensajes_mas_similares(indice_mensaje=299, top_k=30)


MENSAJE DE REFERENCIA
Tipo: pull_request
Titulo: FIX add actual class name to error message in class vs. instance error
Texto: thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ chang see refer issuespr exampl fix see also pleas use keyword fix creat link issu pull request resolv automat close pull request merg see fix relat notic work implementfix explain chang error messag format fstring thus estim name actual resolv test catch fix usag disclosur tool involv creat pleas check box appli make sure adher autom contribut polici use assist code generat write implement fix bug testbenchmark generat document includ exampl research understand comment thank patienc chang scikitlearn requir care attent limit maintain time everi contribut review quick inform tip improv pull request see thank contribut
--------------------------------------------------

,tipo,titulo,texto_limpio,tema,similitud
0,pull_request,Clarify feature scaling effects in k-nearest neighbors example,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
1,pull_request,Teaching pull request - VCS Class - G21,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
2,pull_request,Improve wording in common pitfalls documentation,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
3,pull_request,Fix stratifiedgroupkfold groups lt splits,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
4,pull_request,FIX: Correct response method handling for regressors in _response.py,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
5,pull_request,Fix KNN failure on non-numeric labels with brute algorithm and p=1,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
6,pull_request,Fix issue thread safety buffer,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.961180
7,pull_request,Improve warning message for constant predictors,updat warn messag format constant predictor thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like ...,2,0.961082
8,pull_request,Fix inconsistent string formatting in check_is_fitted function,standard error messag use fstring consist rest file thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain revi...,2,0.959551
9,pull_request,Clarify KNN visualization titles and feature scaling intent,thank contribut pull request pleas ensur taken look contribut guidelin particular follow pull request checklist increas likelihood maintain review like affect user need add changelog entri describ...,2,0.954568
